# Optimización MILP: Resolviendo el TSP en Cabárceno

En este notebook abordamos el núcleo matemático del problema. Modelaremos la visita al parque como un **Problema del Viajante (TSP)** sobre un grafo asimétrico.

Nuestro objetivo es encontrar la matriz de decisión $x_{ij} \in \{0,1\}$ que minimice la distancia total recorrida:

$$\min \sum_{i=1}^{n} \sum_{j=1}^{n} c_{ij} x_{ij}$$

Donde $c_{ij}$ es la distancia real en metros (calculada vía Dijkstra sobre el grafo de OpenStreetMap) entre el recinto $i$ y el recinto $j$. Utilizaremos **Google OR-Tools** para resolver el sistema mediante Programación Lineal Entera Mixta (MILP).

In [1]:
import osmnx as ox
import networkx as nx
import numpy as np
import folium
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
import geopandas as gpd
from shapely.ops import nearest_points


In [2]:
# Cargamos el grafo proyectado en metros (UTM) guardado
G_metros = ox.load_graphml('../data/processed/cabarceno_graph.graphml')

print(f"Grafo cargado. Nodos: {len(G_metros.nodes)}, Aristas: {len(G_metros.edges)}")
print(f"CRS: {G_metros.graph['crs']}")


Grafo cargado. Nodos: 1929, Aristas: 2678
CRS: EPSG:32630


## 1. Mapeo de Recintos a Nodos del Grafo

Definimos las coordenadas GPS (Longitud, Latitud) de los animales que queremos visitar hoy y buscamos el nodo matemático más cercano en nuestra red de carreteras.

In [3]:
# Buscamos los recintos del parque usando los tags que aparecen en OSM:
# praderas (meadow), matorrales (scrub), edificios (building) y atracciones turísticas
tags_recintos = {
    'landuse': ['meadow', 'grass'],
    'natural': 'scrub',
    'building': True,
    'tourism': 'attraction'
}

areas_gdf = ox.features_from_place("Parque de la Naturaleza de Cabárceno, Cantabria, Spain", tags_recintos)

# Nos quedamos solo con los que tienen nombre
recintos_nombrados = areas_gdf[areas_gdf['name'].notna()].copy()

# Si hay nombres duplicados, añadimos sufijo numérico para distinguirlos (ej. "Llama", "Llama (2)")
counts = {}
nombres_unicos = []
for nombre in recintos_nombrados['name']:
    counts[nombre] = counts.get(nombre, 0) + 1
    nombres_unicos.append(nombre)

seen = {}
nombres_finales = []
for nombre in nombres_unicos:
    if counts[nombre] > 1:
        seen[nombre] = seen.get(nombre, 0) + 1
        nombres_finales.append(f"{nombre} ({seen[nombre]})")
    else:
        nombres_finales.append(nombre)

recintos_nombrados = recintos_nombrados.copy()
recintos_nombrados['name'] = nombres_finales

print(f"Recintos con nombre encontrados: {len(recintos_nombrados)}")
print(recintos_nombrados['name'].tolist())


Recintos con nombre encontrados: 40
['Aula de Educación Medioambiental', 'Gorilas', 'Vaca y Caballo Monchino', 'Llama (1)', 'Jaguar', 'Gaur', 'Fauna Ibérica', 'Camello Bactriano', 'Gorila', 'Llama (2)', 'Rinoceronte Blanco', 'Tigre', 'Vaca Tudanca', 'Asno Somalí', 'Cebra', 'Hipopótamo', 'Jirafa, Avestruz y Eland', 'Addax y Búfalo Cafre', 'Bisonte Europeo', 'Cebras Grevy', 'Elefante Africano, Búfalo de agua y Cobo de Lechwe', 'Oryx del Cabo', 'Watusi (1)', 'Hipopótamos pigmeos', 'Oso', 'Cobos de agua', 'León', 'Lobo', 'Papion de Guinea', 'Wallaby de Bennet y Emú', 'Watusi (2)', 'Leones Marinos', 'Aves rapaces', 'Granja', 'Guepardo', 'Linces', 'Reptilario', 'Centro de Recuperación de Fauna Silvestre', 'Exhibiciones Aves rapaces', 'Casa de cebras Grevy']


In [4]:
# Descargamos todos los estacionamientos dentro del parque
parkings_gdf = ox.features_from_place(
    "Parque de la Naturaleza de Cabárceno, Cantabria, Spain",
    tags={'amenity': 'parking'}
)

# Calculamos el centroide de cada parking en UTM
parkings_proj = parkings_gdf.to_crs(G_metros.graph['crs'])
parkings_centroides_utm = parkings_proj.copy()
parkings_centroides_utm.geometry = parkings_proj.geometry.centroid

print(f"Estacionamientos encontrados: {len(parkings_gdf)}")
parkings_centroides_utm.to_crs("EPSG:4326")[['geometry']]


Estacionamientos encontrados: 47


geometry
element_type osmid                                 
node         13098033269  POINT (-3.83712 43.36638)
             13098058550  POINT (-3.83467 43.36840)
             13098060960  POINT (-3.83320 43.36692)
             13098060964  POINT (-3.83414 43.36709)
way          257974639    POINT (-3.83880 43.35482)
             257974640    POINT (-3.82952 43.36255)
             257974641    POINT (-3.83354 43.35685)
             257974642    POINT (-3.82527 43.36016)
             257974643    POINT (-3.85257 43.34429)
             257974645    POINT (-3.82277 43.35894)
             257974646    POINT (-3.84643 43.35303)
             257974647    POINT (-3.83552 43.36500)
             257974648    POINT (-3.83213 43.36586)
             257974649    POINT (-3.83669 43.35737)
             257974650    POINT (-3.82502 43.36549)
             257974651    POINT (-3.85380 43.35002)
             257974652    POINT (-3.85350 43.34937)
             257974654    POINT (-3.84804 43.34932)
             257974655    POINT (-3.82495 43.36511)
             257974656    POINT (-3.82234 43.36376)
             257974658    POINT (-3.85036 43.35154)
             257974660    POINT (-3.83313 43.35606)
             257974661    POINT (-3.83568 43.36321)
             257999737    POINT (-3.83452 43.35660)
             258027006    POINT (-3.85668 43.34390)
             258029365    POINT (-3.84894 43.34912)
             258031759    POINT (-3.82588 43.36013)
             299943072    POINT (-3.83225 43.36149)
             358199856    POINT (-3.83532 43.35645)
             358212384    POINT (-3.84687 43.35046)
             358221430    POINT (-3.83234 43.36194)
             358235832    POINT (-3.83280 43.36651)
             358294677    POINT (-3.85620 43.34947)
             358297032    POINT (-3.85448 43.34531)
             358297033    POINT (-3.85478 43.34420)
             358297309    POINT (-3.84291 43.34881)
             358305808    POINT (-3.83159 43.35975)
             360540221    POINT (-3.82288 43.36378)
             461213505    POINT (-3.83417 43.35511)
             467537056    POINT (-3.83677 43.36505)
             513677049    POINT (-3.81985 43.35925)
             518534804    POINT (-3.84197 43.35042)
             626162416    POINT (-3.83462 43.36780)
             1288899440   POINT (-3.85264 43.35045)
             1308115488   POINT (-3.83279 43.36509)
             1308115489   POINT (-3.83394 43.36802)
             1308115490   POINT (-3.83469 43.36837)

In [5]:
from pyproj import Transformer

# --- Entrada Principal: parking OSM way 358294677 ---
# El índice del GeoDataFrame es un MultiIndex (element_type, osmid)
entrada_centroide_utm = parkings_centroides_utm.loc[('way', 358294677), 'geometry']

# --- Recintos de animales: centroide recinto → parking más cercano (UTM) ---
recintos_proj = recintos_nombrados.to_crs(G_metros.graph['crs'])
recintos_centroides_utm = recintos_proj.geometry.centroid

parking_union = parkings_centroides_utm.geometry.unary_union

waypoints_utm = {}  # nombre -> (x_utm, y_utm) del centroide del parking asignado
for nombre_recinto, centroide_recinto in zip(recintos_nombrados['name'], recintos_centroides_utm):
    _, punto_parking = nearest_points(centroide_recinto, parking_union)
    waypoints_utm[nombre_recinto] = (punto_parking.x, punto_parking.y)

# La Entrada Principal va primero, usando el centroide de su parking en UTM
waypoints_utm = {
    "Entrada Principal": (entrada_centroide_utm.x, entrada_centroide_utm.y),
    **waypoints_utm
}

nombres = list(waypoints_utm.keys())
x_proj = [coords[0] for coords in waypoints_utm.values()]
y_proj = [coords[1] for coords in waypoints_utm.values()]

# Nodo de carretera más cercano a cada centroide de parking (ninguno está en el grafo)
nodos_objetivo = ox.distance.nearest_nodes(G_metros, X=x_proj, Y=y_proj)

# Reconstruimos puntos_interes en lat/lon para los marcadores del mapa
transformer = Transformer.from_crs(G_metros.graph['crs'], "EPSG:4326", always_xy=True)
puntos_interes = {
    nombre: list(transformer.transform(x, y))  # [lon, lat]
    for nombre, (x, y) in waypoints_utm.items()
}

for nombre, nodo in zip(nombres, nodos_objetivo):
    print(f"📍 {nombre} → parking más cercano → nodo ID: {nodo}")


📍 Entrada Principal → parking más cercano → nodo ID: 670744566
📍 Aula de Educación Medioambiental → parking más cercano → nodo ID: 3631612425
📍 Gorilas → parking más cercano → nodo ID: 3631679166
📍 Vaca y Caballo Monchino → parking más cercano → nodo ID: 670799457
📍 Llama (1) → parking más cercano → nodo ID: 3632383628
📍 Jaguar → parking más cercano → nodo ID: 670799465
📍 Gaur → parking más cercano → nodo ID: 5057962905
📍 Fauna Ibérica → parking más cercano → nodo ID: 670786679
📍 Camello Bactriano → parking más cercano → nodo ID: 3631722498
📍 Gorila → parking más cercano → nodo ID: 670776453
📍 Llama (2) → parking más cercano → nodo ID: 670776453
📍 Rinoceronte Blanco → parking más cercano → nodo ID: 9971625188
📍 Tigre → parking más cercano → nodo ID: 3631679166
📍 Vaca Tudanca → parking más cercano → nodo ID: 670799457
📍 Asno Somalí → parking más cercano → nodo ID: 670799465
📍 Cebra → parking más cercano → nodo ID: 3631574507
📍 Hipopótamo → parking más cercano → nodo ID: 5057962905
📍 Jir

## 2. Reducción del Problema: Matriz de Distancias Exactas

El *solver* no necesita conocer todas las curvas de Cabárceno, solo necesita un **grafo completo** con los costes reales $c_{ij}$ entre nuestros 5 puntos de interés. Calculamos estos costes en metros aplicando el algoritmo de **Dijkstra**.

In [6]:
n = len(nodos_objetivo)
distance_matrix = np.zeros((n, n), dtype=int)
paths_dict = {}  # Guardaremos el camino real curvo para reconstruirlo al final

for i, origin in enumerate(nodos_objetivo):
    for j, dest in enumerate(nodos_objetivo):
        if i == j:
            distance_matrix[i][j] = 0
        else:
            length, path = nx.single_source_dijkstra(G_metros, origin, dest, weight='length')
            distance_matrix[i][j] = int(length)
            paths_dict[(i, j)] = path

print("Matriz de distancias (en metros) calculada.")
print(distance_matrix)


Matriz de distancias (en metros) calculada.
[[   0 2343 1462 ...  255 2217 3893]
 [2630    0 2172 ... 2755  122 1967]
 [1871 1989    0 ... 1996 1863 3538]
 ...
 [ 230 2547 1665 ...    0 2421 4096]
 [2684  356 2225 ... 2809    0 2020]
 [3905 1653 3446 ... 4030 1528    0]]


## 3. Resolución del TSP con Google OR-Tools

Instanciamos el modelo de enrutamiento. Configuramos la función de coste y aplicamos la metaheurística de Búsqueda Local Guiada (Guided Local Search) para escapar de mínimos locales y encontrar el orden óptimo de visita.

In [7]:
# 1. Instanciar el gestor (Nodos, Vehículos, Nodo de Inicio que es el 0: Entrada)
manager = pywrapcp.RoutingIndexManager(n, 1, 0)
routing = pywrapcp.RoutingModel(manager)

# 2. Callback de distancias
def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return distance_matrix[from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# 3. Parámetros de búsqueda
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
search_parameters.time_limit.seconds = 2

# 4. Resolver
solution = routing.SolveWithParameters(search_parameters)

if solution:
    print(f"Distancia total de la ruta óptima: {solution.ObjectiveValue()} metros")
    
    optimal_order = []
    index = routing.Start(0)
    while not routing.IsEnd(index):
        optimal_order.append(manager.IndexToNode(index))
        index = solution.Value(routing.NextVar(index))
    optimal_order.append(manager.IndexToNode(index))
    
    print(f"Orden de visita (Índices matriciales): {optimal_order}")
    print(" -> ".join([nombres[i] for i in optimal_order]))
else:
    print("No se encontró solución.")

Distancia total de la ruta óptima: 23013 metros
Orden de visita (Índices matriciales): [0, 8, 25, 15, 1, 34, 39, 33, 24, 35, 28, 22, 30, 29, 27, 36, 31, 19, 26, 40, 20, 18, 23, 21, 17, 7, 9, 2, 12, 13, 3, 5, 14, 10, 16, 6, 11, 4, 37, 32, 38, 0]
Entrada Principal -> Camello Bactriano -> Oso -> Cebra -> Aula de Educación Medioambiental -> Granja -> Exhibiciones Aves rapaces -> Aves rapaces -> Hipopótamos pigmeos -> Guepardo -> Lobo -> Oryx del Cabo -> Wallaby de Bennet y Emú -> Papion de Guinea -> León -> Linces -> Watusi (2) -> Bisonte Europeo -> Cobos de agua -> Casa de cebras Grevy -> Cebras Grevy -> Addax y Búfalo Cafre -> Watusi (1) -> Elefante Africano, Búfalo de agua y Cobo de Lechwe -> Jirafa, Avestruz y Eland -> Fauna Ibérica -> Gorila -> Gorilas -> Tigre -> Vaca Tudanca -> Vaca y Caballo Monchino -> Jaguar -> Asno Somalí -> Llama (2) -> Hipopótamo -> Gaur -> Rinoceronte Blanco -> Llama (1) -> Reptilario -> Leones Marinos -> Centro de Recuperación de Fauna Silvestre -> Entrada P

## 4. Reconstrucción Topológica y Renderizado

El *solver* nos ha devuelto el orden lógico (ej. 0 $\rightarrow$ 3 $\rightarrow$ 1). Ahora recuperamos los nodos intermedios de las carreteras (usando `paths_dict`) para dibujar la ruta poligonal exacta sobre el mapa.

In [10]:
import webbrowser, os
from collections import defaultdict

# Unimos los caminos curvos asegurándonos de no duplicar los nodos de conexión
full_path_nodes = []
for i in range(len(optimal_order) - 1):
    u_idx = optimal_order[i]
    v_idx = optimal_order[i+1]
    
    segment_path = paths_dict[(u_idx, v_idx)]
    if i > 0:
        segment_path = segment_path[1:]
    full_path_nodes.extend(segment_path)

# G_metros preserva 'lat' y 'lon' originales en cada nodo
coordenadas_ruta = [[G_metros.nodes[nodo]['lat'], G_metros.nodes[nodo]['lon']] for nodo in full_path_nodes]

# Dibujamos el mapa interactivo
mapa = folium.Map(location=[43.350, -3.852], zoom_start=14, tiles='cartodbpositron')

# Línea de la ruta
folium.PolyLine(coordenadas_ruta, color="#FF5A5F", weight=5, opacity=0.8).add_to(mapa)

# Mapeamos cada recinto a su número de visita (posición en la ruta, 1-based)
# optimal_order[:-1] excluye el retorno al inicio
nombre_a_orden = {nombres[idx]: pos + 1 for pos, idx in enumerate(optimal_order[:-1])}

# Agrupamos los recintos por coordenadas de parking para evitar marcadores superpuestos
coords_a_nombres = defaultdict(list)
for nombre, coords in puntos_interes.items():
    coords_a_nombres[tuple(coords)].append(nombre)

for coords, nombres_grupo in coords_a_nombres.items():
    es_entrada = "Entrada Principal" in nombres_grupo
    bg_color = '#2ca02c' if es_entrada else '#1f77b4'

    # Si varios recintos comparten parking pueden tener números de visita distintos
    nums = sorted(nombre_a_orden[n] for n in nombres_grupo)
    num_label = "/".join(str(n) for n in nums)

    popup_texto = "<br>".join(f"<b>{n}</b>" for n in nombres_grupo)

    folium.Marker(
        location=[coords[1], coords[0]],
        popup=folium.Popup(popup_texto, max_width=200),
        icon=folium.DivIcon(
            html=f'''<div style="
                background-color: {bg_color};
                color: white;
                border-radius: 50%;
                width: 28px;
                height: 28px;
                display: flex;
                align-items: center;
                justify-content: center;
                font-weight: bold;
                font-size: 12px;
                border: 2px solid white;
                box-shadow: 1px 1px 3px rgba(0,0,0,0.5);
            ">{num_label}</div>''',
            icon_size=(28, 28),
            icon_anchor=(14, 14)
        )
    ).add_to(mapa)

# Guardamos y abrimos el mapa en el navegador
output_path = os.path.abspath('../data/processed/ruta_optima.html')
mapa.save(output_path)
webbrowser.open(f'file://{output_path}')
print(f"Mapa guardado en: {output_path}")


Mapa guardado en: /Users/david/Documents/cabarceno-routing-optimization/data/processed/ruta_optima.html
